# CICIDS2017 Research Execution

Runs the final CICIDS2017 FW-LNSA research profile using natural-distribution sampling and the official machine-learning CSV archive.

Experiment outputs are stored in Google Drive so Colab disconnects do not erase completed runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Repository access

Store a GitHub personal access token in Colab Secrets with the name `GITHUB_TOKEN`. The token needs read access to the private repository. The temporary credential helper avoids placing the token in the Git remote URL.

In [ ]:
import os
import stat
import subprocess
from pathlib import Path
from google.colab import userdata

REPO_URL = "https://github.com/sarosh-jawed/FW-LNSA-NIDS.git"
REPO_DIR = Path("/content/FW-LNSA-NIDS")
TOKEN = userdata.get("GITHUB_TOKEN")
if not TOKEN:
    raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets before continuing.")

askpass = Path("/tmp/fw_lnsa_git_askpass.sh")
askpass.write_text(
    "#!/bin/sh\n"
    "case \"$1\" in\n"
    "  *Username*) echo \"x-access-token\" ;;\n"
    "  *Password*) echo \"$GITHUB_TOKEN\" ;;\n"
    "esac\n"
)
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)

environment = os.environ.copy()
environment["GITHUB_TOKEN"] = TOKEN
environment["GIT_ASKPASS"] = str(askpass)
environment["GIT_TERMINAL_PROMPT"] = "0"

if REPO_DIR.exists():
    subprocess.run(["git", "checkout", "main"], cwd=REPO_DIR, check=True, env=environment)
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True, env=environment)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True, env=environment)

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Git commit:", commit)

In [ ]:
%cd /content/FW-LNSA-NIDS
!python -m pip install -q -r requirements.txt
!python scripts/validate_core_modules.py
!python scripts/validate_fw_lnsa_pipeline.py
!python scripts/validate_cicids2017_pipeline.py
!python scripts/validate_baseline_pipeline.py
!python scripts/validate_research_execution.py

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/FW-LNSA-NIDS')
DATA_ROOT = DRIVE_ROOT / 'data'
RESULT_ROOT = DRIVE_ROOT / 'results'
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Data root:', DATA_ROOT)
print('Persistent result root:', RESULT_ROOT)

## Required archive

Place `MachineLearningCSV.zip` in `MyDrive/FW-LNSA-NIDS/data/cicids2017/`. The code streams the official CSV files directly from the archive.

In [ ]:
CIC_ARCHIVE = DATA_ROOT / 'cicids2017' / 'MachineLearningCSV.zip'
CIC_RAW_DIR = DATA_ROOT / 'cicids2017'
assert CIC_ARCHIVE.exists(), CIC_ARCHIVE
print('Archive size in MB:', CIC_ARCHIVE.stat().st_size / 1024**2)

## Research run

Review the printed retained class distribution and the saved data-quality report before interpreting model metrics.

In [ ]:
!python scripts/run_research_suite.py \
  --profile research \
  --stages cicids2017_fw_lnsa \
  --output-dir "{RESULT_ROOT}" \
  --cic-raw-dir "{CIC_RAW_DIR}" \
  --cic-archive-file "{CIC_ARCHIVE}"

In [ ]:
import pandas as pd
quality = pd.read_csv(RESULT_ROOT / 'tables' / 'cicids2017_data_quality_report.csv')
results = pd.read_csv(RESULT_ROOT / 'tables' / 'cicids2017_fw_lnsa_results.csv')
display(quality)
display(results.groupby(['method_name', 'feature_set'])[['f1', 'recall', 'fpr', 'total_time_sec']].agg(['mean', 'std']))